# 03 · Hardware transfer under one calibration / 单次校准下的硬件迁移

**Goal / 目标**: reproduce point estimates from the public per-circuit tables and preserve sample denominators. / 用公开逐线路表复算点估计并保留样本分母。

Read [case 03](../docs/cases/03-hardware.md) / [中文案例](../docs/cases/03-hardware_zh.md). The tested circuits have six qubits; the platform name is not the algorithm size. / 测试线路使用六比特，平台名不代表算法规模。

In [ ]:
from pathlib import Path
import json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'evidence/E01.json').is_file())
def read(name):
    return json.loads((ROOT / 'evidence' / name).read_text(encoding='utf-8'))


In [ ]:
import csv
from statistics import fmean
with (ROOT / 'evidence/hardware-per-circuit.csv').open(encoding='utf-8', newline='') as stream:
    hardware_rows = list(csv.DictReader(stream))
with (ROOT / 'evidence/cloud-per-circuit.csv').open(encoding='utf-8', newline='') as stream:
    cloud_rows = list(csv.DictReader(stream))
assert len(hardware_rows) == 1000 and len(cloud_rows) == 999
for family in ('message', 'qaoa'):
    rows = [r for r in hardware_rows if r['family'] == family]
    print(family, len(rows), 'depth', fmean(float(r['compiled_depth']) for r in rows),
          'hardware/ideal TV', fmean(float(r['hardware_vs_ideal_tv']) for r in rows))


TV measures distribution difference. Deeper QAOA circuits have much larger hardware/ideal discrepancy in this archive. This is descriptive evidence under one observed calibration, not a controlled causal estimate of depth alone. / TV 衡量分布差异；归档中较深 QAOA 的硬件与理想分布偏差更大。这是单次校准下的描述性观察，不是单独识别深度因果效应的实验。

In [ ]:
for label, rows in [('hardware', hardware_rows), ('cloud-matched', cloud_rows)]:
    qaoa = [r for r in rows if r['family'] == 'qaoa']
    print(label, len(qaoa), 'feasible probability', fmean(float(r['hardware_feasible_k3']) for r in qaoa))
assert sum(int(r['cloud_shots']) for r in cloud_rows) == 1022976
import runpy
print(runpy.run_path(str(ROOT / 'tools/verify_public_evidence.py'))['verify']())


## Exercise / 练习

Explain the 500-versus-499 denominator and why submission batches cannot stand in for independent calibration dates. Propose what new evidence would be needed for a cross-calibration robustness claim. / 解释 500 与 499 的分母差异，以及提交批次为何不能替代独立校准日期；提出跨校准稳健性结论需要的新证据。

The public table supports point-estimate checks. Private job/batch identifiers were removed, so these cells do not reconstruct the archived grouped bootstrap. / 公开表支持点估计复核；私有任务与批次标识已删除，因此这些单元不重建归档分组 bootstrap。